# **Prática - Validação de Dados com Pyndantic**
Com base no vídeo do ArjanCodes, este código utiliza o Pydantic para validar e serializar dados de fontes externas, como APIs ou entradas de usuários. O objetivo é aplicar o princípio de fail fast, detectando erros estruturais o mais cedo possível para evitar que dados inválidos comprometam o restante da aplicação.

Utilizo a BaseModel para definir a estrutura dos dados, empregando tipos como EmailStr para validação de formato e SecretStr para mascarar informações sensíveis com asteriscos. Além disso, a implementação de IntFlag demonstra como gerenciar permissões de forma eficiente através de anotações de tipo.

In [1]:
import sys

if 'google.colab' in sys.modules:
  !pip install 'pydantic[email]'

from enum import auto, IntFlag
from typing import Any
from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    SecretStr,
    ValidationError,
)

class PlayerLevel(IntFlag):
    Noob = auto()
    Pro = auto()
    Legend = auto()
    Staff = Noob | Pro | Legend

class Player(BaseModel):
    nickname: str = Field(examples=["Gamer123"])
    email: EmailStr = Field(
        examples=["player@example.com"],
        description="O email do jogador",
        frozen=True,
    )
    access_key: SecretStr = Field(
        examples=["Key123"], description="Chave de acesso do jogador"
    )
    level: PlayerLevel = Field(default=PlayerLevel.Noob, description="Nivel do jogador")

def validate(data: dict[str, Any]) -> None:
    try:
        player = Player.model_validate(data)
        print(player)
    except ValidationError as e:
        print("Player is invalid")
        for error in e.errors():
            print(error)

def main() -> None:
    good_data = {
        "nickname": "GamerX",
        "email": "contato@gamer.com",
        "access_key": "senha123",
    }
    bad_data = {"email": "email_errado", "access_key": "123"}

    validate(good_data)
    validate(bad_data)

if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 11.2 MB/s eta 0:00:00
nickname='GamerX' email='contato@gamer.com' access_key=SecretStr('**********') level=<PlayerLevel.Noob: 1>
Player is invalid
{'type': 'missing', 'loc': ('nickname',), 'msg': 'Field required', 'input': {'email': 'email_errado', 'access_key': '123'}, 'url': 'https://errors.pydantic.dev/2.12/v/missing'}
{'type': 'value_error', 'loc': ('email',), 'msg': 'value is not a valid email address: An email address must have an @-sign.', 'input': 'email_errado', 'ctx': {'reason': 'An email address must have an @-sign.'}}


Conforme abordado na vídeoaula, o objetivo é garantir que o sistema falhe o mais rápido possível (fail fast) caso os dados não sigam regras estritas de negócio. Utilizo expressões regulares para validar o formato do nickname e a complexidade da chave de acesso, além de implementar um validador de modelo para comparar campos entre si e realizar o hash da senha antes da persistência dos dados. Essa abordagem transforma o Pydantic em uma camada defensiva robusta que sanitiza as informações antes que elas cheguem ao restante do pipeline.

In [2]:
import enum
import hashlib
import re
from typing import Any

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_validator,
    model_validator,
    SecretStr,
    ValidationError,
)

VALID_KEY_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")
VALID_NICKNAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")

class PlayerLevel(enum.IntFlag):
    Noob = 1
    Pro = 2
    Legend = 4
    Staff = 8

class Player(BaseModel):
    nickname: str = Field(examples=["GamerX"])
    email: EmailStr = Field(
        examples=["player@example.com"],
        description="O email do jogador",
        frozen=True,
    )
    access_key: SecretStr = Field(
        examples=["Key12345"], description="Chave de acesso do jogador"
    )
    level: PlayerLevel = Field(
        default=None, description="Nivel do jogador"
    )

    @field_validator("nickname")
    @classmethod
    def validate_nickname(cls, v: str) -> str:
        if not VALID_NICKNAME_REGEX.match(v):
            raise ValueError(
                "Nickname invalido, use apenas letras e no minimo 2 caracteres"
            )
        return v

    @field_validator("level", mode="before")
    @classmethod
    def validate_level(cls, v: int | str | PlayerLevel) -> PlayerLevel:
        op = {int: lambda x: PlayerLevel(x), str: lambda x: PlayerLevel[x], PlayerLevel: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(
                f"Nivel invalido, use: {', '.join([x.name for x in PlayerLevel])}"
            )

    @model_validator(mode="before")
    @classmethod
    def validate_player(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "nickname" not in v or "access_key" not in v:
            raise ValueError("Nickname e access_key sao obrigatorios")
        if v["nickname"].casefold() in v["access_key"].casefold():
            raise ValueError("A chave nao pode conter o nickname")
        if not VALID_KEY_REGEX.match(v["access_key"]):
            raise ValueError(
                "Chave invalida, use 8 caracteres, 1 maiuscula, 1 minuscula e 1 numero"
            )
        v["access_key"] = hashlib.sha256(v["access_key"].encode()).hexdigest()
        return v

def validate(data: dict[str, Any]) -> None:
    try:
        player = Player.model_validate(data)
        print(player)
    except ValidationError as e:
        print("Player is invalid:")
        print(e)

def main() -> None:
    test_data = dict(
        good_data={
            "nickname": "Arjan",
            "email": "example@arjancodes.com",
            "access_key": "Password123",
            "level": "Legend",
        },
        bad_level={
            "nickname": "Arjan",
            "email": "example@arjancodes.com",
            "access_key": "Password123",
            "level": "Iniciante",
        },
        bad_name={
            "nickname": "A1",
            "email": "example@arjancodes.com",
            "access_key": "Password123",
        },
        duplicate={
            "nickname": "Arjan",
            "email": "example@arjancodes.com",
            "access_key": "Arjan12345",
        }
    )

    for case, data in test_data.items():
        print(f"Testando: {case}")
        validate(data)
        print()

if __name__ == "__main__":
    main()

Testando: good_data
nickname='Arjan' email='example@arjancodes.com' access_key=SecretStr('**********') level=<PlayerLevel.Legend: 4>

Testando: bad_level
Player is invalid:
1 validation error for Player
level
  Value error, Nivel invalido, use: Noob, Pro, Legend, Staff [type=value_error, input_value='Iniciante', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

Testando: bad_name
Player is invalid:
1 validation error for Player
nickname
  Value error, Nickname invalido, use apenas letras e no minimo 2 caracteres [type=value_error, input_value='A1', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

Testando: duplicate
Player is invalid:
1 validation error for Player
  Value error, A chave nao pode conter o nickname [type=value_error, input_value={'nickname': 'Arjan', 'em...cess_key': 'Arjan12345'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_err

Neste código, o foco é o controle da saída dos dados, processo conhecido como serialização. De acordo com a vídeoaula, a serialização consiste em transformar a instância de um objeto em outro formato, geralmente JSON, para que possa ser consumido externamente. Aqui, utilizo o model_validator(mode="after") para aplicar regras de negócio em objetos já instanciados e implemento serializers customizados para formatar a saída, garantindo que informações sensíveis sejam omitidas e que tipos complexos sejam convertidos corretamente.

In [3]:
import enum
import hashlib
import re
from typing import Any, Self
from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_serializer,
    field_validator,
    model_serializer,
    model_validator,
    SecretStr,
)

VALID_KEY_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")
VALID_NICKNAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")

class PlayerLevel(enum.IntFlag):
    Noob = 0
    Pro = 1
    Legend = 2
    Staff = 4

class Player(BaseModel):
    nickname: str = Field(examples=["GamerX"])
    email: EmailStr = Field(
        examples=["player@example.com"],
        description="O email do jogador",
        frozen=True,
    )
    access_key: SecretStr = Field(
        examples=["Key12345"],
        description="Chave de acesso do jogador",
        exclude=True
    )
    level: PlayerLevel = Field(
        description="Nivel do jogador",
        default=0,
        validate_default=True,
    )

    @field_validator("nickname")
    def validate_nickname(cls, v: str) -> str:
        if not VALID_NICKNAME_REGEX.match(v):
            raise ValueError("Nickname invalido")
        return v

    @field_validator("level", mode="before")
    @classmethod
    def validate_level(cls, v: int | str | PlayerLevel) -> PlayerLevel:
        op = {int: lambda x: PlayerLevel(x), str: lambda x: PlayerLevel[x], PlayerLevel: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError("Nivel invalido")

    @model_validator(mode="before")
    @classmethod
    def validate_player_pre(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "nickname" not in v or "access_key" not in v:
            raise ValueError("Nickname e access_key sao obrigatorios")
        if v["nickname"].casefold() in v["access_key"].casefold():
            raise ValueError("A chave nao pode conter o nickname")
        if not VALID_KEY_REGEX.match(v["access_key"]):
            raise ValueError("Chave fraca")
        v["access_key"] = hashlib.sha256(v["access_key"].encode()).hexdigest()
        return v

    @model_validator(mode="after")
    def validate_player_post(self) -> Self:
        if self.level == PlayerLevel.Staff and self.nickname != "Arjan":
            raise ValueError("Apenas Arjan pode ser Staff")
        return self

    @field_serializer("level", when_used="json")
    def serialize_level(self, v: PlayerLevel) -> str:
        return v.name

    @model_serializer(mode="wrap", when_used="json")
    def serialize_player(self, serializer, info) -> dict[str, Any]:
        if not info.include and not info.exclude:
            return {"nickname": self.nickname, "level": self.level.name}
        return serializer(self)

def main() -> None:
    data = {
        "nickname": "Arjan",
        "email": "example@arjancodes.com",
        "access_key": "Password123",
        "level": "Staff",
    }

    player = Player.model_validate(data)

    print("Dicionario (model_dump):")
    print(player.model_dump())

    print("\nJSON (model_dump mode='json'):")
    print(player.model_dump(mode="json"))

    print("\nJSON filtrado (sem level):")
    print(player.model_dump(exclude=["level"], mode="json"))

if __name__ == "__main__":
    main()

Dicionario (model_dump):
{'nickname': 'Arjan', 'email': 'example@arjancodes.com', 'level': <PlayerLevel.Staff: 4>}

JSON (model_dump mode='json'):
{'nickname': 'Arjan', 'level': 'Staff'}

JSON filtrado (sem level):
{'nickname': 'Arjan', 'email': 'example@arjancodes.com'}


No código abaixo a prática foca na transição do modelo isolado para uma aplicação web real utilizando FastAPI. Como destacado por Arjan no vídeo, essa integração é o que torna o Pydantic indispensável, permitindo que a API valide automaticamente os dados de entrada antes mesmo de chegarem à lógica de negócio. O uso da configuração extra="forbid" serve como uma trava de segurança essencial para impedir a injeção de atributos não documentados em requisições maliciosas. Além disso, exploramos a automação de campos como IDs e carimbos de data e hora através do default_factory, garantindo que o sistema seja robusto e siga o princípio de falha rápida.

In [4]:
from datetime import datetime
from typing import Optional
from uuid import uuid4

from fastapi import FastAPI
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, EmailStr, Field, field_serializer, UUID4

app = FastAPI()

class Player(BaseModel):
    model_config = {
        "extra": "forbid",
    }
    __storage__ = []

    nickname: str = Field(..., description="Nickname do jogador")
    email: EmailStr = Field(..., description="Email do jogador")
    signup_ts: Optional[datetime] = Field(
        default_factory=datetime.now, kw_only=True
    )
    id: UUID4 = Field(
        default_factory=uuid4, kw_only=True
    )

    @field_serializer("id", when_used="json")
    def serialize_id(self, id: UUID4) -> str:
        return str(id)

@app.get("/players", response_model=list[Player])
async def get_players() -> list[Player]:
    return list(Player.__storage__)

@app.post("/players", response_model=Player)
async def create_player(player: Player):
    Player.__storage__.append(player)
    return player

@app.get("/players/{player_id}", response_model=Player)
async def get_player(player_id: UUID4) -> Player | JSONResponse:
    try:
        return next((p for p in Player.__storage__ if p.id == player_id))
    except StopIteration:
        return JSONResponse(status_code=404, content={"message": "Player not found"})

def main() -> None:
    with TestClient(app) as client:
        for i in range(3):
            response = client.post(
                "/players",
                json={"nickname": f"Gamer_{i}", "email": f"player{i}@example.com"},
            )
            assert response.status_code == 200

        response = client.get("/players")
        print(f"Total de jogadores: {len(response.json())}")

        bad_email = client.post("/players", json={"nickname": "Erro", "email": "invalido"})
        print(f"Status para email invalido: {bad_email.status_code}")

        extra_field = client.post(
            "/players",
            json={"nickname": "Hacker", "email": "hacker@dev.com", "admin": True}
        )
        print(f"Status para campo extra (forbid): {extra_field.status_code}")

if __name__ == "__main__":
    main()

Total de jogadores: 3
Status para email invalido: 422
Status para campo extra (forbid): 422
